# Лабораторная работа №17: RAG для структурированных данных (Text-to-SQL)

**ФИО студента:** _______________________
**Группа:** _______________________

## Цель работы
Изучить подход Text-to-SQL для взаимодействия с реляционными базами данных с использованием больших языковых моделей. Научиться генерировать SQL-запросы на естественном языке, выполнять их и обрабатывать результаты.

## Теоретические сведения
### Что такое Text-to-SQL?
Text-to-SQL — это задача автоматического преобразования вопросов на естественном языке в исполняемые SQL-запросы. Это позволяет пользователям без знаний SQL взаимодействовать с базами данных.

### Архитектура решения
1. **Понимание схемы:** Модель должна знать структуру таблиц (имена колонок, типы данных, связи).
2. **Генерация запроса:** LLM генерирует SQL-код на основе вопроса и схемы.
3. **Валидация и выполнение:** Запрос проверяется на синтаксис и выполняется в БД.
4. **Форматирование ответа:** Результаты запроса переводятся в естественный язык.

### Проблемы и решения
- **Галлюцинации:** Модель может придумать несуществующие колонки. Решение: строгое следование схеме.
- **Сложные запросы:** JOIN, подзапросы требуют хорошего контекста.
- **Безопасность:** Необходимо запрещать деструктивные операции (DROP, DELETE, UPDATE).

## Задание
### Уровень 1 (Базовый)
1. Создайте SQLite базу данных с одной таблицей (например, `employees`).
2. Реализуйте функцию, принимающую вопрос на естественном языке и возвращающую SQL-запрос.
3. Выполните запрос и выведите результат.

### Уровень 2 (Продвинутый)
1. Создайте базу данных с несколькими связанными таблицами (например, `orders`, `customers`, `products`).
2. Реализуйте обработку JOIN-запросов и агрегаций (SUM, COUNT, AVG).
3. Добавьте обработку ошибок (некорректный SQL, пустые результаты).

### Уровень 3 (Индивидуальный)
1. Реализуйте механизм кэширования схем таблиц для ускорения работы.
2. Внедрите проверку безопасности: блокировка опасных запросов.
3. Добавьте возможность уточняющих вопросов к пользователю при неоднозначности запроса.

## Ход работы

### Шаг 1: Установка зависимостей

In [ ]:
%pip install langchain langchain-community langchain-openai sqlite3 faker -q

### Шаг 2: Создание тестовой базы данных

In [ ]:
import sqlite3
from faker import Faker
import random

# Инициализация БД
conn = sqlite3.connect('company.db', check_same_thread=False)
cursor = conn.cursor()

# Создание таблиц
cursor.execute('''
CREATE TABLE IF NOT EXISTS departments (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    budget REAL
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    position TEXT,
    salary REAL,
    department_id INTEGER,
    FOREIGN KEY (department_id) REFERENCES departments(id)
)
''')

# Генерация данных
fake = Faker('ru_RU')
depts = ['IT', 'HR', 'Sales', 'Marketing', 'Finance']

for dept in depts:
    cursor.execute('INSERT INTO departments (name, budget) VALUES (?, ?)', 
                   (dept, random.randint(100000, 1000000)))

for _ in range(50):
    cursor.execute('INSERT INTO employees (name, position, salary, department_id) VALUES (?, ?, ?, ?)',
                   (fake.name(), fake.job(), random.randint(50000, 200000), random.randint(1, 5)))

conn.commit()
print("База данных создана успешно!")

### Шаг 3: Получение схемы базы данных

In [ ]:
def get_db_schema():
    """Получает схему всех таблиц в читаемом формате"""
    schema = ""
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    
    for table in tables:
        table_name = table[0]
        if table_name == 'sqlite_sequence':
            continue
        schema += f"Table: {table_name}\n"
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        for col in columns:
            schema += f"  - {col[1]} ({col[2]})\n"
        schema += "\n"
    return schema

schema = get_db_schema()
print(schema)

### Шаг 4: Настройка LLM для генерации SQL

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

# Для работы требуется API ключ OpenAI или использование локальной модели
# os.environ["OPENAI_API_KEY"] = "your-key-here"

# Используем заглушку для демонстрации (в реальности подключите модель)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

sql_prompt = ChatPromptTemplate.from_messages([
    ("system", """Ты эксперт по SQL. Твоя задача - генерировать только SQL запросы на основе вопроса пользователя и схемы БД.
    Правила:
    1. Используй только предоставленную схему.
    2. Не используй DROP, DELETE, UPDATE, INSERT - только SELECT.
    3. Возвращай ТОЛЬКО SQL код без объяснений и маркеров кода.
    4. Если вопрос не относится к данным, верни 'ERROR: Вопрос не по теме'.
    
    Схема БД:
    {schema}
    """),
    ("human", "{question}")
])

sql_chain = sql_prompt | llm

### Шаг 5: Функция выполнения запроса

In [ ]:
def execute_sql_query(sql_query):
    """Выполняет SQL запрос и возвращает результаты"""
    # Проверка безопасности
    forbidden_keywords = ['DROP', 'DELETE', 'UPDATE', 'INSERT', 'ALTER', 'CREATE']
    if any(keyword in sql_query.upper() for keyword in forbidden_keywords):
        return "Ошибка: Запрещенная операция в запросе!"
    
    try:
        cursor.execute(sql_query)
        results = cursor.fetchall()
        columns = [description[0] for description in cursor.description]
        return {"columns": columns, "data": results}
    except Exception as e:
        return f"Ошибка выполнения запроса: {str(e)}"

### Шаг 6: Пример использования

In [ ]:
# Пример вопроса
question = "Покажи всех сотрудников из отдела IT с зарплатой больше 100000"

# Генерация SQL (в реальном режиме раскомментируйте вызов модели)
# response = sql_chain.invoke({"schema": schema, "question": question})
# sql_query = response.content.strip()

# Для демонстрации используем хардкод запроса
sql_query = """
SELECT e.name, e.position, e.salary, d.name as department
FROM employees e
JOIN departments d ON e.department_id = d.id
WHERE d.name = 'IT' AND e.salary > 100000
"""

print(f"Сгенерированный SQL:\n{sql_query}")
result = execute_sql_query(sql_query)
print(f"\nРезультат: {result}")

### Шаг 7: Интерактивный режим (опционально)

In [ ]:
def ask_database(question):
    """Полный пайплайн: вопрос -> SQL -> ответ"""
    print(f"\nВопрос: {question}")
    
    # В реальной работе здесь будет вызов LLM
    # response = sql_chain.invoke({"schema": schema, "question": question})
    # sql = response.content.strip()
    
    # Демонстрация
    if "сколько" in question.lower() and "сотрудников" in question.lower():
        sql = "SELECT COUNT(*) FROM employees"
    elif "средняя зарплата" in question.lower():
        sql = "SELECT AVG(salary) FROM employees"
    else:
        sql = "SELECT * FROM employees LIMIT 5"
    
    print(f"SQL: {sql}")
    result = execute_sql_query(sql)
    return result

# Тестирование
ask_database("Сколько всего сотрудников в компании?")
ask_database("Какая средняя зарплата?")

## Контрольные вопросы
1. Какие проблемы могут возникнуть при генерации SQL-запросов моделью?
2. Как обеспечить безопасность при выполнении сгенерированных запросов?
3. В чем разница между подходом Text-to-SQL и традиционным RAG?
4. Как обработать ситуацию, когда модель сгенерировала некорректный SQL?

## Выводы
В данной работе изучены принципы работы Text-to-SQL систем. Реализован прототип, позволяющий задавать вопросы к базе данных на естественном языке. Освоены техники защиты от опасных операций и обработки ошибок.